# Part 2 — Work IQ A2A

**A2A (Agent-to-Agent)** is an open standard (Linux Foundation) for agent interoperability. Work IQ
is exposed as an **A2A remote agent** (the *Work IQ Relay Agent*), so any A2A client — regardless of
framework — can delegate a task to it.

Use A2A when **your agent delegates a task** to Work IQ as a peer (multi-agent orchestration).

## 1. Discover the agent via its Agent Card

An **Agent Card** is a JSON document describing identity, endpoint, capabilities, auth, and skills.

In [1]:
import sys, os, requests, json
sys.path.append(os.path.abspath(".."))
from notebooks._shared import get_user_token, WORK_IQ_GATEWAY

token = get_user_token()
# Work IQ publishes a standard A2A agent card at /a2a/.well-known/agent-card.json
card = requests.get(
    f"{WORK_IQ_GATEWAY}/a2a/.well-known/agent-card.json",
    headers={"Authorization": f"Bearer {token}"}, timeout=60,
).json()
print(json.dumps(card, indent=2)[:1200])

{
  "name": "Microsoft Copilot",
  "description": "An AI-powered assistant that helps users with business-related tasks such as managing emails, scheduling meetings, and organizing documents.",
  "url": "https://workiq.svc.cloud.microsoft/a2a",
  "iconUrl": "https://copilot.microsoft.com",
  "provider": {
    "organization": "Microsoft",
    "url": "https://www.microsoft.com"
  },
  "version": "1.0.0",
  "protocolVersion": "0.3.0",
  "capabilities": {
    "streaming": true,
    "pushNotifications": false,
    "stateTransitionHistory": false,
    "extensions": []
  },
  "defaultInputModes": [
    "text"
  ],
  "defaultOutputModes": [
    "text"
  ],
  "skills": [],
  "supportsAuthenticatedExtendedCard": false,
  "additionalInterfaces": [],
  "preferredTransport": "JSONRPC",
  "supportedInterfaces": [
    {
      "url": "https://workiq.svc.cloud.microsoft/a2a",
      "protocolBinding": "JSONRPC",
      "protocolVersion": "1.0"
    }
  ]
}


The card advertises a skill like `ask_work_iq`:

```json
{
  "id": "ask_work_iq",
  "name": "Ask Work IQ",
  "description": "Ask Microsoft 365 Copilot about emails, meetings, files, and other M365 data.",
  "examples": ["What meetings do I have today?", "Summarize my recent emails from my manager"]
}
```

## 2. Send a task over JSON-RPC

A2A calls are JSON-RPC `message/send` requests; the remote agent replies with a task + artifacts.

In [2]:
# A2A message/send is JSON-RPC 2.0. The message object needs kind="message"; Work IQ replies
# over SSE with a `task` whose artifacts carry the answer text.
def a2a_send(text):
    rpc = {"jsonrpc": "2.0", "id": "1", "method": "message/send",
           "params": {"message": {"kind": "message", "role": "user",
                                   "parts": [{"kind": "text", "text": text}], "messageId": "m1"}}}
    r = requests.post(f"{WORK_IQ_GATEWAY}/a2a/",
        headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json",
                 "Accept": "application/json, text/event-stream"}, json=rpc, timeout=120)
    body = r.text
    for line in body.splitlines():
        if line.startswith("data:"):
            body = line[5:].strip(); break
    return json.loads(body)

task = a2a_send("What are my most recent Teams chat messages, and what are they about?")
for art in task["result"].get("artifacts", []):
    for part in art.get("parts", []):
        if part.get("kind") == "text":
            print(part["text"])

I found **11 recent Teams chat message threads**. Here are the most recent ones and what they were about:

### 1. Chat with Refund Hosted (a couple of hours ago)
You asked whether there were any complaints in the mailbox. The response identified a likely complaint: a forwarded email titled “Refund request” from Maria Garcia. You then asked about Maria Garcia's package status, but no tracking details could be found, so the status could not be confirmed. [1](https://teams.microsoft.com/l/message/19:09d2d679-fdea-42a9-aa1e-7eaacb609ea3_1e900dd7-bb6a-49c1-9518-45e056a85220@unq.gbl.spaces/1784024799640?context=%7B%22contextType%22:%22chat%22%7D#fde3e1)

### 2. Chat with refundprocessor (last Thursday)
You asked about weather conditions affecting deliveries in the US. The response discussed localized weather and event-related disruptions affecting some delivery routes. You then asked whether a refund should be issued for Maria Garcia, and the recommendation was **not to issue a refund yet** 

### A2A interaction patterns

- **Task / Message / Artifacts** — a task carries messages and produces artifacts.
- **Streaming** — the card advertises `capabilities.streaming` for incremental results.
- **Context** — the relay keeps the signed-in user's M365 context across turns.

The official `work-iq-samples` repo has full A2A clients in C#, Rust, and Swift.

**Next:** Part 3 — the same power through MCP tools.